## Append, Archive, Load Data
Purpose: To read in the new month's data, append to current database, transform into long format for BI, and archive old data for version control.

In [1]:
# All libraries
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
from datetime import date

# Today's Date for Timestamp
current_date = date.today().strftime("%Y-%m-%d")  # Format: YYYY-MM-DD; Get the current date in the desired format

# File paths
root_folder = '../../../../'
planning_data = os.path.join(root_folder,'planning')
aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')

column_names_location = os.path.join(aware_folder,'1_inputs')
parquet_intermediate_location = os.path.join(aware_folder,'2_planning_processing_files')#'../../../2_planning_processing_files'
database_location = os.path.join(aware_folder,'5_database')#'../../../5_database'


In [2]:
# Read in current database, then save and timestamp old database + archive

# Columns
cols = pd.read_csv(os.path.join(column_names_location,"columns.csv"))["Column Names"].tolist()
cols_p2 = pd.read_csv(os.path.join(column_names_location,"p2_columns.csv"))["Column Names"].tolist()
cols_veh = pd.read_csv(os.path.join(column_names_location,"veh_cols.csv"))["Column Names"].tolist()

# Read in databases
df_database_p1 = pd.read_parquet(os.path.join(database_location,'database_cleaned_p1.parquet'),columns= cols, engine="pyarrow")
df_database_p2 = pd.read_parquet(os.path.join(database_location,'database_cleaned_p2.parquet'),columns= cols_p2, engine="pyarrow")
df_database_veh = pd.read_parquet(os.path.join(database_location,'database_veh.parquet'),engine="pyarrow")

# Archive Databases and timestamp them
df_database_p1.to_parquet(os.path.join(database_location,f"archive/database_cleaned_p1_{current_date}.parquet"), engine="pyarrow", index=False)
df_database_p2.to_parquet(os.path.join(database_location,f"archive/database_cleaned_p2_{current_date}.parquet"), engine="pyarrow", index=False)
df_database_veh.to_parquet(os.path.join(database_location,f"archive/database_veh_{current_date}.parquet"), engine="pyarrow", index=False)

df_database_veh

,ticket,vehicle_plate,vehicle_plate_state,vehicle_type,vendor_tenant_association,driver_name,driver_license_state,control_point_name,ticket_notes,notes,...,arrive_rounded,grant_date,grant_time,grant_rounded,vsc_entry_date,vsc_entry_time,vsc_entry_rounded,gantry_exit_date,gantry_exit_time,gantry_exit_rounded
0,TDTSTBADY,KLN1773,PA,Personally Owned Vehicle (POV),SILVERSTEIN MANAGEMENT,JESUS PAYANO MANZUETA,PA,9A/Liberty,None,Personally Owned Vehicle (POV),...,23:30:00,2025-01-08,23:50:31,23:30:00,None,None,None,None,None,None
1,TKZLPZYOL,BB1845,NY,Access-a-ride,MTA PARA-TRANSIT,Jarrett Matthew Jackson,NY,Trinity,None,None,...,22:00:00,2025-01-08,22:32:12,22:30:00,None,None,None,None,None,None
2,TZWENNUGE,GWU3893,NY,Personally Owned Vehicle (POV),PROPERTY MANAGEMENT,MARK SALMON,NY,9A/Liberty,None,Personally Owned Vehicle (POV),...,21:30:00,2025-01-08,21:51:05,21:30:00,None,None,None,None,None,None
3,TWMSPKVOU,BB2711,NY,Access-a-ride,MTA PARA-TRANSIT,ANIVER SIZAN ...,NY,Trinity,None,Access-a-ride,...,21:00:00,2025-01-08,21:16:06,21:00:00,None,None,None,None,None,None
4,TWVBQPHLY,85194ML,NY,Truck 20'-30',PORT AUTHORITY OF NY AND NJ,"DESMOND,G CLARKE",NY,9A/Liberty,None,Truck 20'-30',...,20:30:00,2025-01-08,20:37:39,20:30:00,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5503,TFFDXVEMG,U76KXF,NJ,Personally Owned Vehicle (POV),PORT AUTHORITY OF NY AND NJ,OZANA STOJANOVIC,NY,9A/Liberty,None,None,...,10:30:00,2025-06-03,10:57:22,10:30:00,None,None,None,None,None,None
5504,THWYFRDPJ,92578MJ,NY,Personally Owned Vehicle (POV),TOUCHTEL ELECTRIC & DATACOM,"JOSEPH,SALVATOR LA ROSA",NY,9A/Liberty,None,None,...,10:00:00,2025-06-03,10:20:42,10:00:00,None,None,None,None,None,None
5505,TGNZFSDAS,KLE7804,NY,Personally Owned Vehicle (POV),SILVERSTEIN MANAGEMENT,JOHN VUKEL,NY,9A/Liberty,None,None,...,09:00:00,2025-06-03,09:04:49,09:00:00,None,None,None,None,None,None
5506,TUYUIMSVJ,LGL1512,NY,Personally Owned Vehicle (POV),THE PORT AUTHORITY OF NEW YORK & NEW JERSEY,PAUL ROZINSKI,NY,9A/Liberty,None,None,...,08:30:00,2025-06-03,08:43:45,08:30:00,None,None,None,None,None,None


In [3]:
# Process all new files and combine them all.
dfs = []
parquet_files = [f for f in os.listdir(os.path.join(parquet_intermediate_location,"p1_sensors_parquet")) if f.endswith('.parquet')]

for file in parquet_files:
    file_path = os.path.join(parquet_intermediate_location,"p1_sensors_parquet", file)
    arch_path = os.path.join(parquet_intermediate_location,"p1_sensors_parquet","processed",file)
    try:
        df = pd.read_parquet(file_path, engine="pyarrow")
        print(f"Successfully read: {file}")
        dfs.append(df)
        print(f"Successfully appended: {file}")
        dest = shutil.move(file_path, arch_path) 
    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")
if dfs:
    # Concatenate all DataFrames into one
    combined_p1_df = pd.concat(dfs, ignore_index=True)
else:
    print("No Parquet files to process.")

Successfully read: WTC_Daily Archive P1_20250607_00-02-59.parquet
Successfully appended: WTC_Daily Archive P1_20250607_00-02-59.parquet
Successfully read: WTC_Daily Archive P1_20250608_00-03-39.parquet
Successfully appended: WTC_Daily Archive P1_20250608_00-03-39.parquet
Successfully read: WTC_Daily Archive P1_20250609_00-08-23.parquet
Successfully appended: WTC_Daily Archive P1_20250609_00-08-23.parquet
No valid data to save.


In [4]:
# Remove data dumps and replace them with NA
if dfs:
    df = combined_p1_df\
        .drop_duplicates(subset=['From'], keep='first')\
        .reset_index(drop = True)
    df.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)

C:\Users\schew\AppData\Local\Temp\ipykernel_34708\2951244096.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      7.0
1      3.0
2      0.0
3      1.0
4      0.0
      ... 
859    2.0
860    1.0
861    2.0
862    2.0
863    4.0
Name: Z11_T2-VeseySt_EastEsc47_OUT, Length: 864, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)
C:\Users\schew\AppData\Local\Temp\ipykernel_34708\2951244096.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      1.0
1      0.0
2      1.0
3      1.0
4      2.0
      ... 
859    1.0
860    0.0
861    0.0
862    1.0
863    2.0
Name: Z11_T2-VeseySt_Stair_IN, Length: 864, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 2:] = d

In [5]:
# Append if new data added
if dfs:
    re_concatenate = [df,df_database_p1]
else:
    re_concatenate = [df_database_p1]

combined_p1_df = pd.concat(re_concatenate, ignore_index=True)
combined_p1_df = combined_p1_df.drop_duplicates().sort_values(by = "From")\
                .reset_index(drop = True)
combined_p1_df

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z27_T3TransitLobby_Stair50_DOWN_IN,Z27_T3TransitLobby_Stair50_UP_OUT,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV
0,2020-02-24 00:00:00,2020-02-24 00:05:00,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0
1,2020-02-24 00:05:00,2020-02-24 00:10:00,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2020-02-24 00:10:00,2020-02-24 00:15:00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
3,2020-02-24 00:15:00,2020-02-24 00:20:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,2020-02-24 00:20:00,2020-02-24 00:25:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
556522,2025-06-08 23:35:00,2025-06-08 23:40:00,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,4.0,1.0,2.0,0.0,0.0
556523,2025-06-08 23:40:00,2025-06-08 23:45:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,4.0,4.0,0.0,2.0,2.0,0.0
556524,2025-06-08 23:45:00,2025-06-08 23:50:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,0.0,2.0,1.0,1.0,8.0,0.0,0.0,4.0,0.0
556525,2025-06-08 23:50:00,2025-06-08 23:55:00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0


In [6]:
# Save new database
combined_p1_df.to_parquet(os.path.join(database_location,"database_cleaned_p1.parquet"), engine="pyarrow", index=False)

In [7]:
# Process all new files and combine them all.
dfs = []
parquet_files = [f for f in os.listdir(os.path.join(parquet_intermediate_location,"p2_sensors_parquet")) if f.endswith('.parquet')]

for file in parquet_files:
    file_path = os.path.join(parquet_intermediate_location,"p2_sensors_parquet", file)
    arch_path = os.path.join(parquet_intermediate_location,"p2_sensors_parquet","processed",file)
    try:
        df = pd.read_parquet(file_path, engine="pyarrow")
        print(f"Successfully read: {file}")
        dfs.append(df)
        print(f"Successfully appended: {file}")
        dest = shutil.move(file_path, arch_path) 
    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")
if dfs:
    # Concatenate all DataFrames into one
    combined_p2_df = pd.concat(dfs, ignore_index=True)
else:
    print("No Parquet files to process.")

Successfully read: WTC_Daily Archive P2_20250606_00-02-58.parquet
Successfully appended: WTC_Daily Archive P2_20250606_00-02-58.parquet
Successfully read: WTC_Daily Archive P2_20250607_00-03-01.parquet
Successfully appended: WTC_Daily Archive P2_20250607_00-03-01.parquet
Successfully read: WTC_Daily Archive P2_20250608_00-03-41.parquet
Successfully appended: WTC_Daily Archive P2_20250608_00-03-41.parquet
Successfully read: WTC_Daily Archive P2_20250609_00-08-25.parquet
Successfully appended: WTC_Daily Archive P2_20250609_00-08-25.parquet
No valid data to save.


In [8]:
# Remove data dumps and replace them with NA
if dfs:
    df_p2 = combined_p2_df\
        .drop_duplicates(subset=['From'], keep='first')\
        .reset_index(drop = True)
    df_p2.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)

# Append if new data added
if dfs:
    re_concatenate = [df_p2,df_database_p2]
else:
    re_concatenate = [df_database_p2]

combined_p2_df = pd.concat(re_concatenate, ignore_index=True)
combined_p2_df = combined_p2_df.drop_duplicates().sort_values(by = "From")\
                .reset_index(drop = True)

combined_p2_df.iloc[:, 2:] = combined_p2_df.iloc[:, 2:].mask(combined_p2_df.iloc[:, 2:] > 500, np.nan)

combined_p2_df

C:\Users\schew\AppData\Local\Temp\ipykernel_34708\3241813777.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      0
1      0
2      0
3      0
4      0
      ..
283    0
284    0
285    0
286    0
287    0
Name: Z08_HEC1M-ELEV18_IN, Length: 288, dtype: int64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_p2.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)
C:\Users\schew\AppData\Local\Temp\ipykernel_34708\3241813777.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      1
1      0
2      0
3      0
4      0
      ..
283    0
284    0
285    0
286    0
287    1
Name: Z08_HEC1M-ELEV18_OUT, Length: 288, dtype: int64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_p2.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)
C:\Us

,From,To Time,Z08_HEC1M-ELEV18_IN,Z08_HEC1M-ELEV18_OUT,Z33_C1-WestConcourse_IN,Z33_C1-WestConcourse_OUT,Z32_T1ESC19&20/ZONE32_NorthEsc_IN,Z32_T1ESC19&20/ZONE32_NorthEsc_OUT,Z32_T1ESC19&20/ZONE32_SouththEsc_IN,Z32_T1ESC19&20/ZONE32_SouththEsc_OUT,...,Z19_PATHturnstiles-East_249-255(south)_IN,Z19_PATHturnstiles-East_249-255(south)_OUT,Z19_PATHturnstiles-East_256-261(center)_IN,Z19_PATHturnstiles-East_256-261(center)_OUT,Z19_PATHturnstiles-East_262-267(center)_IN,Z19_PATHturnstiles-East_262-267(center)_OUT,Z19_PATHturnstiles-East_268-274(north)_IN,Z19_PATHturnstiles-East_268-274(north)_OUT,Z20_PATHturnstiles-SE_Passageway-North_NB,Z20_PATHturnstiles-SE_Passageway-North_SB
0,2023-10-01 00:00:00,2023-10-01 00:05:00,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,17.0,1.0,13.0,1.0,9.0,0.0,7.0,3.0,0.0
1,2023-10-01 00:05:00,2023-10-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,14.0,0.0,9.0,1.0,12.0,0.0,18.0,5.0,3.0
2,2023-10-01 00:10:00,2023-10-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,18.0,0.0,1.0,5.0,16.0,2.0,3.0
3,2023-10-01 00:15:00,2023-10-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,26.0,18.0,47.0,21.0,19.0,12.0,3.0,23.0,5.0,0.0
4,2023-10-01 00:20:00,2023-10-01 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,3.0,17.0,1.0,8.0,0.0,11.0,1.0,14.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133339,2025-06-08 23:35:00,2025-06-08 23:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133340,2025-06-08 23:40:00,2025-06-08 23:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133341,2025-06-08 23:45:00,2025-06-08 23:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133342,2025-06-08 23:50:00,2025-06-08 23:55:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Save new database
combined_p2_df.to_parquet(os.path.join(database_location,"database_cleaned_p2.parquet"), engine="pyarrow", index=False)

In [10]:
different_cols = combined_p2_df.columns.difference(combined_p1_df.columns)

combined_p2_df = combined_p2_df[different_cols]

required_cols = cols[:2]  # ['From', 'To']

# Convert Index to list and prepend the required columns
combined_cols = required_cols + different_cols.tolist()
combined_cols
# Use these columns to create the combined dataframe
combined_p2_df = df_database_p2[combined_cols]

# Make sure 'From' and 'To' are in both DataFrames (they already are)
# Merge on them explicitly
combined_df = pd.merge(
    combined_p1_df,
    combined_p2_df,
    how='left',
    on=['From', 'To Time']
)



In [11]:
# Combine the first two columns with the filtered data
combined_df["Time_Bucket"] = combined_df["From"].dt.floor("30T")  # "30T" means 30-minute interval
grouped_df = combined_df.groupby("Time_Bucket").sum(numeric_only=True).reset_index()
long_df = grouped_df.melt(id_vars=["Time_Bucket"], var_name="Location", value_name="Count")
long_df['date'] = long_df['Time_Bucket'].dt.date#.astype('datetime64[ns]')
long_df['time'] = long_df['Time_Bucket'].dt.time
long_df = long_df.drop(columns=['Time_Bucket'])
long_df.to_parquet(os.path.join(database_location,"database_cleaned_long.parquet"), engine="pyarrow", index=False)
long_df

C:\Users\schew\AppData\Local\Temp\ipykernel_34708\996857127.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  combined_df["Time_Bucket"] = combined_df["From"].dt.floor("30T")  # "30T" means 30-minute interval


,Location,Count,date,time
0,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,00:00:00
1,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,00:30:00
2,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,01:00:00
3,Z01_T4-ChurchSt_RevDoor_IN,2.0,2020-02-24,01:30:00
4,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,02:00:00
...,...,...,...,...
24073915,Z43_SouthWestPassage_OUT,0.0,2025-06-08,21:30:00
24073916,Z43_SouthWestPassage_OUT,0.0,2025-06-08,22:00:00
24073917,Z43_SouthWestPassage_OUT,0.0,2025-06-08,22:30:00
24073918,Z43_SouthWestPassage_OUT,0.0,2025-06-08,23:00:00


In [12]:
# Process all new files and combine them all.
dfs = []
parquet_files = [f for f in os.listdir(os.path.join(parquet_intermediate_location,"veh_parquet")) if f.endswith('.parquet')]

for file in parquet_files:
    file_path = os.path.join(parquet_intermediate_location,"veh_parquet", file)
    arch_path = os.path.join(parquet_intermediate_location,"veh_parquet","processed",file)
    try:
        df = pd.read_parquet(file_path, engine="pyarrow")
        print(f"Successfully read: {file}")
        dfs.append(df)
        print(f"Successfully appended: {file}")
        dest = shutil.move(file_path, arch_path) 
    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")
if dfs:
    # Concatenate all DataFrames into one
    combined_veh_df = pd.concat(dfs, ignore_index=True)
else:
    print("No Parquet files to process.")

No valid data to save.
No Parquet files to process.


In [15]:
# Append if new data added
if dfs:
    re_concatenate = [df,df_database_veh]
else:
    re_concatenate = [df_database_veh]

combined_veh_df = pd.concat(re_concatenate, ignore_index=True)
combined_veh_df = combined_veh_df.drop_duplicates()\
                .sort_values(by=["arrive_date", "arrive_time"])\
                .reset_index(drop = True)
combined_veh_df
combined_veh_df.to_parquet(os.path.join(database_location,"database_veh.parquet"), engine="pyarrow", index=False)

combined_veh_df

,ticket,vehicle_plate,vehicle_plate_state,vehicle_type,vendor_tenant_association,driver_name,driver_license_state,control_point_name,ticket_notes,notes,...,arrive_rounded,grant_date,grant_time,grant_rounded,vsc_entry_date,vsc_entry_time,vsc_entry_rounded,gantry_exit_date,gantry_exit_time,gantry_exit_rounded
0,TYQMNKWDX,HKD4734,NY,Personally Owned Vehicle (POV),MSA,EMMANUEL DELAROSA,NY,9A/Liberty,None,Personally Owned Vehicle (POV),...,02:30:00,2025-01-02,02:51:00,02:30:00,None,None,None,None,None,None
1,TMSIUCNLG,KEN4733,NY,Personally Owned Vehicle (POV),SILVERSTEIN MANAGEMENT,JOHN VUKEL,NY,9A/Liberty,None,None,...,06:00:00,2025-01-02,06:07:00,06:00:00,None,None,None,None,None,None
2,TWLZURWGG,BB2321,NY,Access-a-ride,MTA PARA-TRANSIT,PRINCE STEELE ...,NY,Trinity,None,None,...,06:00:00,2025-01-02,06:11:00,06:00:00,None,None,None,None,None,None
3,TGTQIOPQF,92578MJ,NY,Personally Owned Vehicle (POV),TOUCHTEL ELECTRIC & DATACOM,"JOSEPH,SALVATOR LA ROSA",NY,9A/Liberty,None,None,...,06:00:00,2025-01-02,06:13:00,06:00:00,None,None,None,None,None,None
4,TANTCYVXM,FSC2429,NY,Personally Owned Vehicle (POV),SILVERSTEIN MANAGEMENT,JAMES SCHOEPFER ...,NY,9A/Liberty,None,Personally Owned Vehicle (POV),...,06:00:00,2025-01-02,06:20:00,06:00:00,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5503,TCULVFSTJ,BH3455,None,None,None,TERRELL JONES ...,NY,Trinity,None,None,...,01:00:00,2025-06-05,01:16:54,01:00:00,None,None,None,None,None,None
5504,TZONUDIUK,85194ML,NY,Truck 20'-30',PORT AUTHORITY OF NY AND NJ,JOEL ABREU ...,NY,9A/Liberty,None,None,...,01:00:00,2025-06-05,01:26:02,01:00:00,None,None,None,None,None,None
5505,TEOOFZHOH,58819MJ,None,None,None,EVERALDO MENDES ...,NY,9A/Liberty,None,None,...,02:30:00,2025-06-05,02:45:53,02:30:00,None,None,None,None,None,None
5506,TGZSKNZFB,P11UEA,NJ,Personally Owned Vehicle (POV),PROPERTY MANAGEMENT,JOHN BARDAZZI,NJ,9A/Liberty,None,None,...,02:30:00,2025-06-05,02:58:22,02:30:00,None,None,None,None,None,None
